In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/11 13:45:52 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/11 13:45:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-041b1a69-ff41-408f-bb76-0267c3ec001a;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:
# Cell 0 — Optional: ensure Spark & paths helpers are available
# (same pattern you use in other notebooks)

import sys
import importlib
import os

# Adjust PROJECT_ROOT if needed
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Add Silver utils into path
silver_dir = os.path.join(PROJECT_ROOT, "_02_Silver")
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)

print("Loaded utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
✅ utils_silver.py loaded
Loaded utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 — Intro markdown (keep as comments for .py style)

"""
ML Monitoring Overview

Goal:
- Read the central ML monitoring Delta table in Gold
- Show latest metrics for every (use_case, model_name)
- Identify the best model per use_case by AUC
- Give a time-series style view so you can see metric drift over time

The table is written by utils_silver.write_ml_monitoring and stored in:
  paths_gold["_ml_monitoring"]
"""


# Cell 2 — Imports & paths

In [3]:


from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Your storage account (same as everywhere else)
STORAGE_ACCOUNT = "clientdatastorage"

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(STORAGE_ACCOUNT)

# This key is what write_ml_monitoring uses
ML_MONITORING_PATH = paths_gold["ml_monitoring"]

print("ML monitoring Delta path:", ML_MONITORING_PATH)


ML monitoring Delta path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring


#  Cell 3 — Load ML monitoring table & basic sanity checks

In [4]:


from pyspark.sql import functions as F

from pyspark.sql.utils import AnalysisException

try:
    ml_mon = (
        spark.read
             .format("delta")
             .load(ML_MONITORING_PATH)
    )
except AnalysisException as e:
    print("⚠️ Could not read ML monitoring table.")
    print("   Make sure you've run the model notebooks with write_ml_monitoring first.")
    raise e

print(f"Total rows in ml_monitoring: {ml_mon.count()}")
ml_mon.printSchema()

# sort by latest run
ml_mon.orderBy(F.col("run_ts").desc()).show(20, truncate=False)


25/12/11 13:46:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ml_monitoring: 12
root
 |-- run_ts: timestamp (nullable = true)
 |-- model_name: string (nullable = true)
 |-- use_case: string (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- dataset_split: string (nullable = true)
 |-- auc: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- precision: double (nullable = true)
 |-- recall: double (nullable = true)
 |-- f1: double (nullable = true)
 |-- notes: string (nullable = true)



+--------------------------+----------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|run_ts                    |model_name            |use_case        |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                                             |
+--------------------------+----------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|2025-12-10 21:38:47.401751|LogisticRegression    |claims_fraud    |ft_claims_risk |test         |0.9869242149815263|0.9906162752321401|1.0               |0.9742365243797908 |0.9869501575408738|fraud model sweep                                 |
|2025-12-10 21:3

# ✅ Cell 4 — Latest metrics per (use_case, model_name)

In [5]:
# Cell 4 – Latest metrics per (use_case, model_name)

from pyspark.sql import Window
from pyspark.sql import functions as F

# For each (use_case, model_name), keep only the most recent run_ts
w_latest = Window.partitionBy("use_case", "model_name").orderBy(F.col("run_ts").desc())

latest = (
    ml_mon
        .withColumn("rn_latest", F.row_number().over(w_latest))
        .filter(F.col("rn_latest") == 1)
        .drop("rn_latest")
)

print("Latest metrics per (use_case, model_name):")
latest.orderBy("use_case", "model_name").show(truncate=False)


Latest metrics per (use_case, model_name):


+--------------------------+----------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+----------------------+
|run_ts                    |model_name            |use_case        |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                 |
+--------------------------+----------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+----------------------+
|2025-12-10 21:38:47.401751|LogisticRegression    |claims_fraud    |ft_claims_risk |test         |0.9869242149815263|0.9906162752321401|1.0               |0.9742365243797908 |0.9869501575408738|fraud model sweep     |
|2025-12-10 21:38:47.401751|RandomForest          |claims_fraud    |ft_claims_risk |test         |0.987404471063458 |0.704724865

# ✅ Cell 5 — Best current model per use_case (by AUC)

In [6]:
# Cell 5 – Best current model per use_case (by AUC)

# Starting from the 'latest' DF from Cell 4
w_auc = Window.partitionBy("use_case").orderBy(F.col("auc").desc())

best_current = (
    latest
        .withColumn("rn_auc", F.row_number().over(w_auc))
        .filter(F.col("rn_auc") == 1)
        .drop("rn_auc")
)

print("Best current model per use_case (by AUC):")
best_current.orderBy("use_case").show(truncate=False)


Best current model per use_case (by AUC):


+--------------------------+------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+---------------------+
|run_ts                    |model_name        |use_case        |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                |
+--------------------------+------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+---------------------+
|2025-12-10 21:38:47.401751|RandomForest      |claims_fraud    |ft_claims_risk |test         |0.987404471063458 |0.7047248659786458|1.0               |0.18930766781769648|0.3183493606243436|fraud model sweep    |
|2025-12-10 21:35:57.969517|LogisticRegression|high_cost_claims|ft_claims_risk |test         |0.998845032858229 |0.9643650375973383|0.99824679703304

# ✅ Cell 6 — Time-series of AUC for a given use_case (e.g. policy_churn)

In [7]:
# Cell 6 – Time series of AUC for a given use_case

USE_CASE = "policy_churn"   # change to "claims_fraud", "high_cost_claim", etc as needed

ts_policy = (
    ml_mon
        .filter(F.col("use_case") == USE_CASE)
        .select("run_ts", "model_name", "dataset_name", "dataset_split", "auc", "accuracy", "f1")
        .orderBy(F.col("run_ts").desc())
)

print(f"AUC time series for use_case = {USE_CASE}:")
ts_policy.show(50, truncate=False)


AUC time series for use_case = policy_churn:
+--------------------------+----------------------+---------------+-------------+------------------+--------+---+
|run_ts                    |model_name            |dataset_name   |dataset_split|auc               |accuracy|f1 |
+--------------------------+----------------------+---------------+-------------+------------------+--------+---+
|2025-12-10 20:09:15.242522|LogisticRegression    |ft_policy_churn|test         |0.9999997460547502|1.0     |1.0|
|2025-12-10 20:09:15.242522|GBTClassifier         |ft_policy_churn|test         |0.9999999999999999|1.0     |1.0|
|2025-12-10 20:09:15.242522|RandomForestClassifier|ft_policy_churn|test         |0.9999999999999999|1.0     |1.0|
|2025-12-10 19:28:35.388188|RandomForestClassifier|ft_policy_churn|test         |0.9999999999999999|1.0     |1.0|
|2025-12-10 19:28:35.388188|LogisticRegression    |ft_policy_churn|test         |0.9999997460547502|1.0     |1.0|
+--------------------------+---------------

# ✅ Cell 7 — Simple “degradation” check (AUC drop vs previous run)

In [8]:
# Cell 7 – Degradation check (AUC drop compared to previous run)

w_by_time = (
    Window
        .partitionBy("use_case", "model_name")
        .orderBy(F.col("run_ts").asc())
)

with_lag = (
    ml_mon
        .withColumn("prev_auc", F.lag("auc").over(w_by_time))
        .withColumn("delta_auc", F.col("auc") - F.col("prev_auc"))
)

# Flag potential issues where AUC has dropped more than 0.02
degraded = (
    with_lag
        .filter(F.col("prev_auc").isNotNull())
        .filter(F.col("delta_auc") < -0.02)
        .orderBy(F.col("run_ts").desc())
)

print("Runs where AUC dropped by more than 0.02 vs previous run:")
degraded.select(
    "run_ts", "use_case", "model_name", "dataset_name",
    "dataset_split", "prev_auc", "auc", "delta_auc"
).show(truncate=False)


Runs where AUC dropped by more than 0.02 vs previous run:


+------+--------+----------+------------+-------------+--------+---+---------+
|run_ts|use_case|model_name|dataset_name|dataset_split|prev_auc|auc|delta_auc|
+------+--------+----------+------------+-------------+--------+---+---------+
+------+--------+----------+------------+-------------+--------+---+---------+



# ✅ Cell 8 — Persist a compact view for BI (bupa_gold.ml_monitoring_view)

In [9]:
DB_GOLD   = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ml_monitoring_view
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view'
""")

DataFrame[]

In [11]:
# Cell 8 — Persist ml_monitoring compact view as a Delta table

DB_GOLD    = "bupa_gold"
TABLE_NAME = "ml_monitoring_view"

# Drop any existing TABLE or VIEW with this name
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.{TABLE_NAME}")
spark.sql(f"DROP VIEW  IF EXISTS {DB_GOLD}.{TABLE_NAME}")

# Write out the compact table as Delta
(
    ml_mon
      .select(
          "run_ts",
          "use_case",
          "model_name",
          "dataset_name",
          "dataset_split",
          "auc",
          "accuracy",
          "precision",
          "recall",
          "f1",
          "notes",
      )
      .write.format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{DB_GOLD}.{TABLE_NAME}")
)

print(f"✅ Created Delta table: {DB_GOLD}.{TABLE_NAME}")
spark.table(f"{DB_GOLD}.{TABLE_NAME}").show(20, truncate=False)


✅ Created Delta table: bupa_gold.ml_monitoring_view
+--------------------------+----------------+----------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|run_ts                    |use_case        |model_name            |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                                             |
+--------------------------+----------------+----------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|2025-12-10 19:28:35.388188|policy_churn    |LogisticRegression    |ft_policy_churn|test         |0.9999997460547502|1.0               |0.0               |0.0                |1.0               |best threshold =

In [12]:
df = spark.table(f"{DB_GOLD}.{TABLE_NAME}")
df.show(20, truncate=False)

+--------------------------+----------------+----------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|run_ts                    |use_case        |model_name            |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                                             |
+--------------------------+----------------+----------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|2025-12-10 19:28:35.388188|policy_churn    |LogisticRegression    |ft_policy_churn|test         |0.9999997460547502|1.0               |0.0               |0.0                |1.0               |best threshold = 0.5                              |
|2025-12-10 20:0

In [14]:


ML_MON_PATH="abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view"

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ML_MON_PATH)

print("✅ Delta files written to ADLS")

✅ Delta files written to ADLS


In [18]:
df = spark.read.format("delta").load(
    "abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view"
)

print(f"Data loaded from ADLS Path: {ML_MON_PATH} {df.printSchema(), df.show(20, truncate=False)}")

root
 |-- run_ts: timestamp (nullable = true)
 |-- use_case: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- dataset_split: string (nullable = true)
 |-- auc: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- precision: double (nullable = true)
 |-- recall: double (nullable = true)
 |-- f1: double (nullable = true)
 |-- notes: string (nullable = true)

+--------------------------+----------------+----------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------------------------------------+
|run_ts                    |use_case        |model_name            |dataset_name   |dataset_split|auc               |accuracy          |precision         |recall             |f1                |notes                                             |
+--------------------------+----------------+--------------------